# **Q2: Part - A**

In [ ]:
def Rc4Ksa(KeyBytes):
    # I follow the clarified assignment constraint: key length must be 4 to 16 bytes
    if not (4 <= len(KeyBytes) <= 16):
        raise ValueError("Key length must be between 4 and 16 bytes.")

    # I start with the identity permutation for the RC4 state array S
    State = list(range(256))
    JIndex = 0
    KeyLen = len(KeyBytes)

    # I permute State using the Key Scheduling Algorithm (KSA)
    for IIndex in range(256):
        JIndex = (JIndex + State[IIndex] + KeyBytes[IIndex % KeyLen]) & 255
        State[IIndex], State[JIndex] = State[JIndex], State[IIndex]

    # I return the permuted state and reset PRGA indices to 0,0
    return State, 0, 0


def Rc4Prga(State, IIndex, JIndex, Count):
    # I generate 'Count' keystream bytes using PRGA while updating i and j each step
    Out = []
    for _ in range(Count):
        IIndex = (IIndex + 1) & 255
        JIndex = (JIndex + State[IIndex]) & 255
        State[IIndex], State[JIndex] = State[JIndex], State[IIndex]
        KByte = State[(State[IIndex] + State[JIndex]) & 255]
        Out.append(KByte)

    # I return keystream bytes along with updated indices
    return bytes(Out), IIndex, JIndex


def Rc4Keystream(KeyBytes, Count):
    # I run KSA once per message/key and then use PRGA to obtain the keystream
    State, IIndex, JIndex = Rc4Ksa(KeyBytes)
    KeystreamBytes, _, _ = Rc4Prga(State, IIndex, JIndex, Count)
    return KeystreamBytes


def Rc4Crypt(KeyBytes, DataBytes):
    # I generate keystream equal to data length and XOR to encrypt/decrypt
    KeystreamBytes = Rc4Keystream(KeyBytes, len(DataBytes))
    return bytes([D ^ K for D, K in zip(DataBytes, KeystreamBytes)])


KeyHex = "54657374"
KeyBytes = bytes.fromhex(KeyHex)
PlaintextBytes = b"Hello World"

print("Original Key (hexadecimal):", KeyHex)
print("Original Key (ASCII):", KeyBytes.decode("utf-8", errors="replace"))
print("Original Plaintext:", PlaintextBytes.decode("utf-8", errors="replace"))

# I print the first 16 keystream bytes as required in Part A
Keystream16 = Rc4Keystream(KeyBytes, 16)
print("\nFirst 16 Keystream Bytes (hexadecimal):", Keystream16.hex())

# I encrypt plaintext to produce ciphertext
CiphertextBytes = Rc4Crypt(KeyBytes, PlaintextBytes)
print("CipherText (hexadecimal):", CiphertextBytes.hex())

# I decrypt ciphertext by running RC4 again with the same key
RecoveredBytes = Rc4Crypt(KeyBytes, CiphertextBytes)
RecoveredText = RecoveredBytes.decode("utf-8", errors="replace")
print("Decrypted PlainText:", RecoveredText)

# I check if decryption exactly matches original plaintext bytes
IsMatch = (RecoveredBytes == PlaintextBytes)
print("\nWhether Original PlainText Matches Decrypted PlainText:", IsMatch)

Original Key (hexadecimal): 54657374
Original Key (ASCII): Test
Original Plaintext: Hello World

First 16 Keystream Bytes (hexadecimal): 7a6e4d48423b0e4a74b7bb08539b9bb5
CipherText (hexadecimal): 320b21242d1b592506dbdf
Decrypted PlainText: Hello World

Whether Original PlainText Matches Decrypted PlainText: True


# **Q2: Part - B**

In [4]:
from secrets import token_bytes

def RunRc4BiasExperiment(MessageBytes):
    Trials = 50000
    ZeroCount = 0
    MsgLen = len(MessageBytes)

    # I need at least 2 bytes of keystream to talk about the "second byte"
    if MsgLen < 2:
        raise ValueError("MessageBytes must be at least 2 bytes long.")

    # I repeat the experiment for many random keys
    for _ in range(Trials):
        KeyBytes = token_bytes(16)  # I use 16-byte keys as required in Part B
        KeyLen = len(KeyBytes)

        # I run KSA to initialize the RC4 permutation State
        State = list(range(256))
        JIndex = 0
        for IIndex in range(256):
            JIndex = (JIndex + State[IIndex] + KeyBytes[IIndex % KeyLen]) & 255
            State[IIndex], State[JIndex] = State[JIndex], State[IIndex]

        # I run PRGA for the full message length
        IIndex = 0
        JIndex = 0
        Keystream = []
        for _ in range(MsgLen):
            IIndex = (IIndex + 1) & 255
            JIndex = (JIndex + State[IIndex]) & 255
            State[IIndex], State[JIndex] = State[JIndex], State[IIndex]
            KByte = State[(State[IIndex] + State[JIndex]) & 255]
            Keystream.append(KByte)

        # I record only the second keystream byte for the bias experiment
        SecondByte = Keystream[1]
        if SecondByte == 0:
            ZeroCount += 1

    ObservedProb = ZeroCount / Trials
    UniformProb = 1 / 256
    Rc4TheoryProb = 2 / 256

    print("Trials Done:", Trials)
    print("Message Length Used (bytes):", MsgLen)
    print("Second Byte Being 0 Counts:", ZeroCount)
    print("Observed Probability:", ObservedProb)
    print("\nReference Uniform Random Probability:", UniformProb)
    print("Reference RC4 Theoretical (approx):", Rc4TheoryProb)

# Example: use the same message as Part A
RunRc4BiasExperiment(b"Hello World")

Trials Done: 50000
Message Length Used (bytes): 11
Second Byte Being 0 Counts: 394
Observed Probability: 0.00788

Reference Uniform Random Probability: 0.00390625
Reference RC4 Theoretical (approx): 0.0078125


# **Q3: Part - 1**

In [ ]:
def Hex80ToBits(HexStr):
    # I convert a 20-hex-character string (80 bits) into a list of 80 bits (0/1)
    HexStr = HexStr.lower().replace("0x", "")
    if len(HexStr) != 20:
        raise ValueError("80-bit hex must be exactly 20 hex characters.")
    Value = int(HexStr, 16)
    Bits = [(Value >> (79 - i)) & 1 for i in range(80)]
    return Bits

def Bytes80ToBits(ByteStr):
    # I convert exactly 10 bytes (80 bits) into a list of 80 bits (0/1)
    if len(ByteStr) != 10:
        raise ValueError("80-bit value must be exactly 10 bytes.")
    Value = int.from_bytes(ByteStr, "big")
    Bits = [(Value >> (79 - i)) & 1 for i in range(80)]
    return Bits

def BitsToHex(Bits):
    # I convert a bit list into hex (useful to show a compact view of keystream bits)
    BitStr = "".join(str(b) for b in Bits)
    PadLen = (4 - (len(BitStr) % 4)) % 4
    BitStr = BitStr + ("0" * PadLen)
    HexLen = len(BitStr) // 4
    Value = int(BitStr, 2)
    return format(Value, "0{}x".format(HexLen))

def InitTriviumState(KeyBits, IvBits):
    # I initialize the 288-bit Trivium state using the standard specification layout
    State = [0] * 288

    # I load the 80-bit key into s1..s80, then set the remaining bits of R1 to 0
    State[0:80] = KeyBits[:]
    State[80:93] = [0] * 13

    # I load the 80-bit IV into s94..s173, then set the remaining bits of R2 to 0
    State[93:93+80] = IvBits[:]
    State[93+80:177] = [0] * 4

    # I set R3 as 108 zeros followed by three 1s at the end (s286=s287=s288=1)
    State[177:285] = [0] * 108
    State[285] = 1
    State[286] = 1
    State[287] = 1

    return State

def TriviumClock(State):
    # I compute the output bit z_i from the current state before shifting
    ZBit = State[65] ^ State[161] ^ State[242]  # s66 ^ s162 ^ s243

    # I compute the three nonlinear update values t1, t2, t3 using XOR and AND terms
    T1 = State[65] ^ (State[90] & State[91]) ^ State[92] ^ State[170]
    T2 = State[161] ^ (State[174] & State[175]) ^ State[176] ^ State[263]
    T3 = State[242] ^ (State[285] & State[286]) ^ State[287] ^ State[68]

    # I shift Register 1 (s1..s93) and insert t3 into s1
    for i in range(92, 0, -1):
        State[i] = State[i - 1]
    State[0] = T3

    # I shift Register 2 (s94..s177) and insert t1 into s94
    for i in range(176, 93, -1):
        State[i] = State[i - 1]
    State[93] = T1

    # I shift Register 3 (s178..s288) and insert t2 into s178
    for i in range(287, 177, -1):
        State[i] = State[i - 1]
    State[177] = T2

    # I return the output bit for this clock cycle
    return ZBit

def TriviumKeystream(KeyInput, IvInput, NumBits=1000, InputType="hex"):
    # I accept input either as 80-bit hex strings (20 hex chars) or as 10-byte values
    if InputType == "hex":
        KeyBits = Hex80ToBits(KeyInput)
        IvBits = Hex80ToBits(IvInput)
    elif InputType == "bytes":
        KeyBits = Bytes80ToBits(KeyInput)
        IvBits = Bytes80ToBits(IvInput)
    else:
        raise ValueError("InputType must be 'hex' or 'bytes'.")

    # I initialize the 288-bit internal state
    State = InitTriviumState(KeyBits, IvBits)

    # I run the 1152-cycle initialization phase (4 * 288) and discard output bits
    for _ in range(1152):
        _ = TriviumClock(State)

    # I generate the required number of keystream bits after initialization
    OutBits = []
    for _ in range(NumBits):
        OutBits.append(TriviumClock(State))

    return OutBits

def PrintBits(Bits, PerLine=100):
    # I print the bitstream in fixed-width lines so it is easy to paste into the report
    BitStr = "".join(str(b) for b in Bits)
    for i in range(0, len(BitStr), PerLine):
        print(BitStr[i:i+PerLine])

KeyHex = "a1b2c3d4e5f60718293a"
IvHex  = "0f1e2d3c4b5a69788796"

KeyBits = Hex80ToBits(KeyHex)
IvBits = Hex80ToBits(IvHex)

print("Trivium Input Details:")
print("Key (hex):", KeyHex)
print("IV  (hex):", IvHex)
print("Key length (bits):", len(KeyBits))
print("IV length  (bits):", len(IvBits))
print("Initialization cycles discarded:", 1152)
print("Keystream bits requested:", 1000)

# I generate the first 1000 keystream bits after the initialization phase
Bits = TriviumKeystream(KeyHex, IvHex, NumBits=1000, InputType="hex")

# I compute basic summary statistics of the generated keystream
OnesCount = sum(Bits)
ZerosCount = len(Bits) - OnesCount
RatioOnes = OnesCount / len(Bits)

print("\nKeystream Summary:")
print("Total bits generated:", len(Bits))
print("Number of 1s:", OnesCount)
print("Number of 0s:", ZerosCount)
print("Fraction of 1s:", RatioOnes)
print("First 64 bits (preview):", "".join(str(b) for b in Bits[:64]))
print("First 128 bits (hex view):", BitsToHex(Bits[:128]))

print("\nFirst 1000 keystream bits after initialization:")
PrintBits(Bits, PerLine=100)

Trivium Input Details:
Key (hex): a1b2c3d4e5f60718293a
IV  (hex): 0f1e2d3c4b5a69788796
Key length (bits): 80
IV length  (bits): 80
Initialization cycles discarded: 1152
Keystream bits requested: 1000

Keystream Summary:
Total bits generated: 1000
Number of 1s: 529
Number of 0s: 471
Fraction of 1s: 0.529
First 64 bits (preview): 1110011100101101101001100010100100111000010010010101100100110001
First 128 bits (hex view): e72da62938495931115be1d67d5c16f9

First 1000 keystream bits after initialization:
1110011100101101101001100010100100111000010010010101100100110001000100010101101111100001110101100111
1101010111000001011011111001000101100000011001100101111111111001101110010101110101111101011110111101
0000001010001110011001101001010110110101011100000101000111010011101100111111101101111110010001010001
1101111101110110011100000011100101001001011110101111001011101011011100110000010101010011011011100111
00010110011010010110111001001111001111100101001001001010101011011101111110001110101011101000